In [ ]:
import copy
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA    = Path().resolve().parent.parent / 'data' / 'processed'
TS_BASE = ('https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/'
           'end-use-load-profiles-for-us-building-stock/2025/resstock_amy2018_release_1/'
           'timeseries_individual_buildings/by_state/upgrade=0')

PARC     = 'nn_buildings_elargi.csv'
N_BAT    = None
N_OUT    = 4
HIDDEN   = 128
SEQ_LEN  = 168
EPOQUES  = 40
PATIENCE = 8
GRAINE   = 42

In [ ]:
class LoadNet(nn.Module):
    def __init__(self, n_time, n_static, hidden=128, n_out=4):
        super().__init__()
        self.gru = nn.GRU(n_time, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2 + n_static, hidden), nn.ReLU(),
            nn.Linear(hidden, n_out),
            nn.Softplus(),
        )

    def forward(self, x_time, static):
        h, _ = self.gru(x_time)
        s = static.unsqueeze(1).expand(-1, h.size(1), -1)
        return self.head(torch.cat([h, s], dim=-1))

In [ ]:
WEA = ['out.outdoor_air_drybulb_temp..c', 'out.outdoor_air_relative_humidity..percentage',
       'out.weather.wind_speed..meter_per_second',
       'out.weather.direct_normal_solar_radiation..watt_per_m2',
       'out.weather.diffuse_solar_radiation..watt_per_m2']
SCHED = ['out.schedules.' + s for s in [
    'occupants', 'vacancy', 'lighting_interior', 'lighting_garage', 'plug_loads_other',
    'plug_loads_tv', 'clothes_dryer', 'clothes_washer', 'dishwasher', 'cooking_range',
    'ceiling_fan', 'hot_water_fixtures', 'hot_water_clothes_washer', 'hot_water_dishwasher',
    'no_space_cooling', 'no_space_heating']]
SETP = ['out.schedules.heating_setpoint..c', 'out.schedules.cooling_setpoint..c']
TGT  = ['out.electricity.' + t + '.energy_consumption..kwh' for t in
        ['total', 'heating', 'cooling', 'hot_water']]
NOMS = ['total', 'chauffage', 'clim', 'eau_chaude']


def load_building(bldg_id, state, cop_bat=1.0):
    path = DATA / f'{bldg_id}-0.parquet'
    ts = pd.read_parquet(path if path.exists() else f'{TS_BASE}/state={state}/{bldg_id}-0.parquet')
    if not path.exists():
        ts.to_parquet(path)
    ts['timestamp'] = pd.to_datetime(ts['timestamp'])
    brut = ts.set_index('timestamp').reindex(columns=WEA + SCHED + SETP + TGT)

    h = brut[WEA + SCHED + SETP].resample('1h').mean().iloc[:8760]
    y = brut[TGT].resample('1h').sum().iloc[:8760]
    h[SCHED] = h[SCHED].fillna(0.0)
    if h[SETP].isna().any().any():
        raise ValueError(f'bâtiment {bldg_id} : consignes absentes. '
                         'Relancer extraction_timeseries_oedi avec FORCE = True.')

    i = h.index
    cal = pd.DataFrame({
        'h_sin': np.sin(2*np.pi*i.hour/24),      'h_cos': np.cos(2*np.pi*i.hour/24),
        'd_sin': np.sin(2*np.pi*i.dayofweek/7),  'd_cos': np.cos(2*np.pi*i.dayofweek/7),
        'm_sin': np.sin(2*np.pi*(i.month-1)/12), 'm_cos': np.cos(2*np.pi*(i.month-1)/12),
    }, index=i)

    t_ext = h[WEA[0]]
    ecart = pd.DataFrame({
        'ecart_chauffage': (h[SETP[0]] - t_ext).clip(lower=0),
        'ecart_clim':      (t_ext - h[SETP[1]]).clip(lower=0),
    }, index=i)
    cop_t = np.clip(cop_bat * (0.6 + 0.02 * t_ext), 1.0, max(cop_bat, 1.0))
    ecart['besoin_elec_chauffage'] = ecart['ecart_chauffage'] / cop_t

    return pd.concat([h[WEA + SCHED + SETP], cal, ecart], axis=1), y

In [ ]:
_parc = pd.read_csv(DATA / PARC)
if N_BAT is not None:
    _parc = _parc.sample(n=N_BAT, random_state=0)
BUILDINGS = list(_parc.itertuples(index=False, name=None))

_eff  = pd.read_parquet(DATA / 'metadata_clean.parquet',
                        columns=['bldg_id', 'in.hvac_heating_efficiency']).set_index('bldg_id')
_hspf = _eff['in.hvac_heating_efficiency'].astype(str).str.extract(r'([\d.]+)\s*HSPF')[0].astype(float)
COP   = (_hspf / 3.412).fillna(1.0).to_dict()


def build_dataset(buildings, L=SEQ_LEN):
    Xs, Ys, bids = [], [], []
    for bid, st in buildings:
        f, y = load_building(bid, st, COP.get(bid, 1.0))
        n = len(f) // L
        Xs.append(f.values[:n*L].reshape(n, L, -1))
        Ys.append(y.values[:n*L].reshape(n, L, -1))
        bids += [bid] * n
    return np.concatenate(Xs), np.concatenate(Ys), np.array(bids)


X_time, Y, bids = build_dataset(BUILDINGS)
print(f'parc : {len(BUILDINGS)} bâtiments | {_parc["in.state"].nunique()} états')
print('X_time', X_time.shape, '| Y', Y.shape, '| fenêtres', len(bids))

In [ ]:
preds = pd.read_parquet(DATA / 'static_preds_oos.parquet')
if 'bldg_id' in preds.columns:
    preds = preds.set_index('bldg_id')
assert list(preds.columns) == NOMS, list(preds.columns)

feat = pd.read_parquet(DATA / 'X_47features.parquet')
feat = feat.assign(tau=feat['C'] / (feat['UA'] + feat['H_ve']) / 3.6)
feat = feat.drop(columns=[c for c in feat.columns if 'setpoint' in c])
assert set(bids) <= set(feat.index)

equip = pd.read_parquet(DATA / 'metadata_clean.parquet',
                        columns=['bldg_id', 'in.hvac_cooling_type', 'in.water_heater_fuel',
                                 'in.hvac_heating_efficiency']).set_index('bldg_id')
a_clim = (equip.loc[bids, 'in.hvac_cooling_type'] != 'None').values.astype('float32')
ecs_el = (equip.loc[bids, 'in.water_heater_fuel'] == 'Electricity').values.astype('float32')

hspf = equip.loc[bids, 'in.hvac_heating_efficiency'].astype(str).str.extract(
    r'([\d.]+)\s*HSPF')[0].astype(float)
pac  = hspf.notna().values.astype('float32')
cop  = (hspf / 3.412).fillna(1.0).values.astype('float32')
mshp = equip.loc[bids, 'in.hvac_heating_efficiency'].astype(str).str.startswith(
    'MSHP').values.astype('float32')

S_brut = np.hstack([preds.loc[bids].values, feat.loc[bids].values,
                    a_clim[:, None], ecs_el[:, None],
                    pac[:, None], cop[:, None], mshp[:, None]]).astype('float32')

un   = np.ones_like(a_clim)
GATE = np.stack([un, un, a_clim, ecs_el], axis=-1).astype('float32')

rng   = np.random.default_rng(GRAINE)
uniq  = np.unique(bids)
val_b = set(rng.choice(uniq, size=max(1, round(0.2 * len(uniq))), replace=False))
val   = np.array([b in val_b for b in bids])
tr    = ~val


def fit_std(a):
    ax = tuple(range(a.ndim - 1))
    return a.mean(ax, keepdims=True), a.std(ax, keepdims=True) + 1e-8


xm, xs = fit_std(X_time[tr]); Xn = (X_time - xm) / xs
sm, ss = fit_std(S_brut[tr]); Sn = (S_brut - sm) / ss
ys = Y[tr].std(tuple(range(Y.ndim - 1)), keepdims=True) + 1e-8
Yn = Y / ys

print(f'train {tr.sum()} fenêtres | val {val.sum()} | bâtiments val {len(val_b)}/{len(uniq)}')
print(f'f(t) = {Xn.shape[-1]} entrées | s = {Sn.shape[1]} colonnes')
print(f'ratio bâtiments train / colonnes de s = {(len(uniq) - len(val_b)) / Sn.shape[1]:.1f}')
print(f'sans clim : {int((a_clim == 0).sum())} fenêtres | sans ECS électrique : {int((ecs_el == 0).sum())}')
print(f'min(Yn) = {Yn.min():.3f} | moyenne(Xn) = {Xn.mean():.2e}')

In [ ]:
torch.manual_seed(GRAINE)
to_t = lambda a: torch.tensor(a, dtype=torch.float32)
masque = lambda p, g: p * g.unsqueeze(1)

loader = DataLoader(TensorDataset(to_t(Xn[tr]), to_t(Sn[tr]), to_t(Yn[tr]), to_t(GATE[tr])),
                    batch_size=64, shuffle=True)
Xva, Sva = to_t(Xn[val]).to(device), to_t(Sn[val]).to(device)
Yva, Gva = to_t(Yn[val]).to(device), to_t(GATE[val]).to(device)

model = LoadNet(Xn.shape[-1], Sn.shape[1], HIDDEN, N_OUT).to(device)
opt, lossf = torch.optim.Adam(model.parameters(), lr=1e-3), nn.MSELoss()

best, best_state, best_ep = float('inf'), None, -1
for epoch in range(EPOQUES):
    model.train()
    for xb, sb, yb, gb in loader:
        xb, sb, yb, gb = xb.to(device), sb.to(device), yb.to(device), gb.to(device)
        opt.zero_grad()
        lossf(masque(model(xb, sb), gb), yb).backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        vloss = lossf(masque(model(Xva, Sva), Gva), Yva).item()
    print(f'epoch {epoch:3d}  val_loss {vloss:.4f}')

    if vloss < best:
        best, best_ep, best_state = vloss, epoch, copy.deepcopy(model.state_dict())
    elif epoch - best_ep >= PATIENCE:
        break

model.load_state_dict(best_state)
print(f'meilleure val_loss {best:.4f} (époque {best_ep})')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

model.eval()
with torch.no_grad():
    pred = masque(model(Xva, Sva), Gva).cpu().numpy() * ys
true = Y[val]
bva  = bids[val]

print('=== Validation (bâtiments jamais vus) ===')
for i, n in enumerate(NOMS):
    p, t = pred[..., i].ravel(), true[..., i].ravel()
    print(f'{n:11} R²={r2_score(t, p):6.3f}   RMSE={np.sqrt(((p-t)**2).mean()):.3f} kWh/h')

for i, (n, g) in enumerate([('clim', GATE[val][:, 2]), ('eau_chaude', GATE[val][:, 3])], start=2):
    absent = g == 0
    if absent.any():
        print(f'  {n:11} {int(absent.sum())} fenêtres sans équipement -> '
              f'max prédit = {pred[absent, :, i].max():.1e}')

nmbe, cvrmse = {n: [] for n in NOMS}, {n: [] for n in NOMS}
for b in np.unique(bva):
    k = bva == b
    for i, n in enumerate(NOMS):
        t, p = true[k, :, i], pred[k, :, i]
        if t.sum() > 0:
            nmbe[n].append((p.sum() - t.sum()) / t.sum() * 100)
            cvrmse[n].append(np.sqrt(((p - t) ** 2).mean()) / t.mean() * 100)

print(f"\n{'usage':11} {'n':>4} {'|NMBE| médian':>14} {'CV(RMSE) médian':>17} {'part <10%':>11}")
for n in NOMS:
    a = np.abs(nmbe[n])
    print(f'{n:11} {len(a):4d} {np.median(a):13.1f}% {np.median(cvrmse[n]):16.1f}% '
          f'{(a < 10).mean()*100:10.0f}%')

ordre = np.argsort(np.abs(nmbe['total']))
batiments = np.unique(bva)[[ordre[len(ordre)//2], ordre[-1]]]
fig, axes = plt.subplots(2, 2, figsize=(13, 6))
for ligne, b in enumerate(batiments):
    w = np.flatnonzero(bva == b)[0]
    for col, i in enumerate([0, 1]):
        ax = axes[ligne, col]
        ax.plot(true[w, :, i], label='réel', lw=1.5)
        ax.plot(pred[w, :, i], label='prédit', lw=1.5, alpha=0.8)
        ax.set(title=f'bâtiment {b} — {NOMS[i]}', xlabel='heure', ylabel='kWh')
        ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Réel vs prédit — cas médian (haut) et pire biais (bas)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
SEM_HIVER, SEM_ETE = 2, 28

bat_val = np.unique(bva)
an_reel = np.array([true[bva == b, :, 0].sum() for b in bat_val])
an_pred = np.array([pred[bva == b, :, 0].sum() for b in bat_val])
nmbe_tot = (an_pred - an_reel) / an_reel * 100
cv_tot = np.array([np.sqrt(((pred[bva == b, :, 0] - true[bva == b, :, 0]) ** 2).mean())
                   / true[bva == b, :, 0].mean() * 100 for b in bat_val])

diag = pd.DataFrame({'bldg_id': bat_val, 'annuel_kWh': an_reel, 'predit_kWh': an_pred,
                     'nmbe_%': nmbe_tot, 'cvrmse_%': cv_tot}).set_index('bldg_id')

print('=== 5 plus fortes consommations ===')
print(diag.sort_values('annuel_kWh', ascending=False).head(5).round(1).to_string())
print('\n=== 5 plus gros biais en valeur absolue ===')
print(diag.reindex(diag['nmbe_%'].abs().sort_values(ascending=False).index)
          .head(5).round(1).to_string())

In [ ]:
def semaine(b, sem):
    k = np.flatnonzero(bva == b)
    return k[sem] if sem < len(k) else k[-1]


choix = [
    (diag['annuel_kWh'].idxmax(), 'plus forte consommation'),
    (diag.index[int(np.argmin((diag['nmbe_%'] - diag['nmbe_%'].median()).abs().to_numpy()))],
     'cas médian'),
    (diag['nmbe_%'].abs().idxmax(), 'plus gros biais'),
]

fig, axes = plt.subplots(len(choix), 2, figsize=(14, 3.1 * len(choix)))
for ligne, (b, etiquette) in enumerate(choix):
    ymax = 0
    for col, (sem, saison) in enumerate([(SEM_HIVER, 'hiver'), (SEM_ETE, 'été')]):
        w = semaine(b, sem)
        ymax = max(ymax, true[w, :, 0].max(), pred[w, :, 0].max())
    for col, (sem, saison) in enumerate([(SEM_HIVER, 'hiver'), (SEM_ETE, 'été')]):
        w = semaine(b, sem)
        ax = axes[ligne, col]
        ax.plot(true[w, :, 0], color='#2a78d6', lw=1.4, label='réel')
        ax.plot(pred[w, :, 0], color='#eb6834', lw=1.4, ls='--', label='prédit')
        ax.fill_between(range(168), true[w, :, 0], pred[w, :, 0], color='#eb6834', alpha=.15)
        ax.set(ylim=(0, ymax * 1.15), xlim=(0, 167), xlabel='heure de la semaine',
               ylabel='kWh/h' if col == 0 else None)
        ax.set_title(f'{b} — {etiquette} — {saison}   '
                     f'(réel {true[w, :, 0].sum():.0f} / prédit {pred[w, :, 0].sum():.0f} kWh)',
                     fontsize=10)
        ax.set_xticks(range(0, 169, 24)); ax.grid(alpha=.3)
        if ligne == 0 and col == 0:
            ax.legend()
plt.suptitle('Semaine d\'hiver et semaine d\'été — consommation totale', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
lim = [0, max(an_reel.max(), an_pred.max()) * 1.05]
ax.scatter(an_reel, an_pred, s=26, color='#2a78d6', alpha=.6, edgecolors='none')
ax.plot(lim, lim, 'k--', lw=1)
ax.set(xlim=lim, ylim=lim, xlabel='consommation annuelle réelle (kWh)',
       ylabel='prédite (kWh)', title='Annuel reconstitué par bâtiment')
ax.grid(alpha=.3)

ax = axes[1]
ax.scatter(an_reel, nmbe_tot, s=26, color='#2a78d6', alpha=.6, edgecolors='none')
ax.axhline(0, color='black', lw=.9)
for s in (-10, 10):
    ax.axhline(s, color='#8A5A00', lw=.9, ls=':')
ax.set(xlabel='consommation annuelle réelle (kWh)', ylabel='NMBE (%)',
       title='Biais en fonction de la taille du bâtiment')
ax.grid(alpha=.3)

ax = axes[2]
q = pd.qcut(an_reel, min(4, len(an_reel)), labels=False, duplicates='drop')
grp = pd.DataFrame({'q': q, 'cv': cv_tot, 'nmbe': np.abs(nmbe_tot)}).groupby('q')
x = np.arange(grp.ngroups)
ax.bar(x - .19, grp['cv'].median(), width=.36, color='#2a78d6', label='CV(RMSE)')
ax.bar(x + .19, grp['nmbe'].median(), width=.36, color='#eb6834', label='|NMBE|')
ax.axhline(30, color='#8A5A00', lw=.9, ls=':')
ax.set_xticks(x)
ax.set_xticklabels([f'Q{i+1}' for i in x])
ax.set(xlabel='quartile de consommation annuelle', ylabel='%',
       title='Erreur médiane par quartile de taille')
ax.legend(); ax.grid(axis='y', alpha=.3)

plt.suptitle('Diagnostic de l\'erreur — 101 bâtiments de validation', fontsize=13)
plt.tight_layout(); plt.show()

r = np.corrcoef(an_reel, np.abs(nmbe_tot))[0, 1]
print(f'corrélation consommation annuelle / |NMBE| : {r:+.2f}')
print(f'|NMBE| médian  petits bâtiments (Q1) : {np.median(np.abs(nmbe_tot)[q == 0]):.1f}%'
      f'   gros bâtiments (Q{grp.ngroups}) : {np.median(np.abs(nmbe_tot)[q == q.max()]):.1f}%')

In [ ]:
COLONNES = list(load_building(*BUILDINGS[0])[0].columns)
uniq_ids = list(dict.fromkeys(bids))
prem     = {b: np.where(bids == b)[0][0] for b in uniq_ids}

torch.save({
    'state_dict': model.state_dict(),
    'archi'    : 'gru_bidirectionnel',
    'tete'     : 'additive',
    'n_time'   : Xn.shape[-1], 'n_static': Sn.shape[1],
    'hidden'   : HIDDEN,       'n_out'   : N_OUT,
    'xm': xm, 'xs': xs, 'sm': sm, 'ss': ss, 'ys': ys,
    'colonnes' : COLONNES,
    'feats'    : ['cop', 'copt', 'oos'],
    'buildings': BUILDINGS,
    'val_b'    : sorted(int(b) for b in val_b),
    'gate'     : {int(b): GATE[prem[b]].tolist() for b in uniq_ids},
    'statique' : {int(b): S_brut[prem[b]].tolist() for b in uniq_ids},
}, DATA / 'loadnet.pt')

print('modèle sauvegardé ->', DATA / 'loadnet.pt')
print(f'  f(t) = {Xn.shape[-1]} | s = {Sn.shape[1]} | {len(BUILDINGS)} bâtiments')